# 1.03 线性代数

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/01_prerequisites/1.03_linear_algebra.ipynb)


## 为什么要学

不夸张地说，**深度学习就是大量的矩阵乘法加一点点非线性**：

- 全连接层 `y = Wx + b` 是矩阵乘法；
- 注意力 `softmax(QKᵀ/√d)V` 是矩阵乘法加"向量相似度"；
- 强化学习里的贝尔曼方程 `v = r + γPv` 是一个线性方程组；
- 微调大模型用的 LoRA 是"低秩矩阵分解"。

本课用代码 + 图像建立直觉，参考 [d2l 2.3 线性代数](https://zh.d2l.ai/chapter_preliminaries/linear-algebra.html) 与附录。

---

## 1. 向量与点积

向量就是一列数：$\mathbf{x} = [x_1, x_2, \dots, x_n]^\top$。两个向量的**点积**（内积）：

$$
\mathbf{x}\cdot\mathbf{y} = \mathbf{x}^\top\mathbf{y} = \sum_{i=1}^n x_i y_i = \|\mathbf{x}\|\,\|\mathbf{y}\|\cos\theta
$$

几何意义：**点积衡量两个向量"方向有多一致"**。$\theta$ 是夹角：同向时 $\cos\theta = 1$，垂直时为 0，反向时为 −1。

- **余弦相似度** $\cos\theta = \frac{\mathbf{x}^\top\mathbf{y}}{\|\mathbf{x}\|\|\mathbf{y}\|}$：只看方向、不看长度。CLIP/SigLIP 就是用它判断"图像和文字是否匹配"。
- **注意力分数** $\mathbf{q}^\top\mathbf{k}$：query 和 key 越"同向"，注意力越大。

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

x = torch.tensor([1.0, 2.0, 3.0])
y = torch.tensor([2.0, 4.0, 6.1])
z = torch.tensor([3.0, 0.0, -1.0])
cos = lambda a, b: (a @ b) / (a.norm() * b.norm())
print("x·y =", (x @ y).item(), " cos(x,y) =", cos(x, y).item())   # 几乎同向 -> 接近 1
print("x·z =", (x @ z).item(), " cos(x,z) =", cos(x, z).item())   # 垂直 -> 0

### 范数：向量的"长度"

$$
\|\mathbf{x}\|_2 = \sqrt{\textstyle\sum_i x_i^2},\qquad \|\mathbf{x}\|_1 = \textstyle\sum_i |x_i|,\qquad \|\mathbf{x}\|_\infty = \max_i |x_i|
$$

- L2 范数：欧氏距离；MSE 损失就是 L2 距离的平方。
- L1 范数：ACT 用 L1 损失训练动作（对离群值更鲁棒）。

In [ ]:
v = torch.tensor([3.0, -4.0])
print(v.norm(p=2), v.norm(p=1), v.norm(p=float("inf")))

---

## 2. 矩阵 = 线性变换

矩阵 $A\in\mathbb{R}^{m\times n}$ 作用在向量上 $\mathbf{y} = A\mathbf{x}$，相当于把 $n$ 维空间**线性地**变到 $m$ 维空间：拉伸、旋转、剪切、投影。

一个关键视角：**$A\mathbf{x}$ 是 $A$ 的各列按 $\mathbf{x}$ 的分量加权求和**：

$$
A\mathbf{x} = x_1 \mathbf{a}_{:,1} + x_2 \mathbf{a}_{:,2} + \dots + x_n \mathbf{a}_{:,n}
$$

下面画出几个 2×2 矩阵把单位正方形变成什么：

In [ ]:
square = torch.tensor([[0, 0], [1, 0], [1, 1], [0, 1], [0, 0]], dtype=torch.float32).T  # (2, 5) 每列一个点
mats = {
    "scale": torch.tensor([[2.0, 0], [0, 0.5]]),
    "rotate 45°": torch.tensor([[np.cos(np.pi/4), -np.sin(np.pi/4)], [np.sin(np.pi/4), np.cos(np.pi/4)]], dtype=torch.float32),
    "shear": torch.tensor([[1.0, 1.0], [0, 1.0]]),
    "projection (rank 1)": torch.tensor([[1.0, 1.0], [1.0, 1.0]]) / 2,
}
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, (name, A) in zip(axes, mats.items()):
    out = A @ square
    ax.plot(square[0], square[1], "k--", label="original")
    ax.plot(out[0], out[1], "r-", lw=2, label="A @ x")
    ax.set_title(name); ax.set_aspect("equal"); ax.set_xlim(-1, 2.5); ax.set_ylim(-1, 2.5); ax.grid(alpha=.3)
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

最后一个"投影"矩阵把整个平面压到了一条线上：**它的秩（rank）是 1**。信息丢失了，变换不可逆。

### 矩阵乘法的形状规则

$$
(m\times n)\cdot(n\times p) = (m\times p)
$$

"中间相同的维度被消掉"。计算量约为 $m\cdot n\cdot p$ 次乘加。所以注意力的 $QK^\top$ 在序列长度 $L$ 上是 $O(L^2)$ 的，这就是长序列贵的原因。

In [ ]:
Q = torch.randn(300, 64); K = torch.randn(300, 64)
print((Q @ K.T).shape)    # (300, 300)：每个 token 对每个 token 的分数

---

## 3. 逆矩阵与线性方程组

方阵 $A$ 可逆时，方程 $A\mathbf{x} = \mathbf{b}$ 的解是 $\mathbf{x} = A^{-1}\mathbf{b}$。实际计算时**不要显式求逆**，用 `torch.linalg.solve`，更快也更稳定。

> **预告**：第 2 章贝尔曼方程 $\mathbf{v} = \mathbf{r} + \gamma P\mathbf{v}$ 移项得 $(I - \gamma P)\mathbf{v} = \mathbf{r}$。这就是一个线性方程组！只要 $\gamma<1$，$I-\gamma P$ 一定可逆。

In [ ]:
A = torch.tensor([[3.0, 1.0], [1.0, 2.0]])
b = torch.tensor([9.0, 8.0])
x = torch.linalg.solve(A, b)
print("解:", x, " 验证 A@x =", A @ x)

---

## 4. 特征值与特征向量

如果 $A\mathbf{v} = \lambda\mathbf{v}$（$\mathbf{v}\ne 0$），则 $\mathbf{v}$ 是**特征向量**、$\lambda$ 是**特征值**：$A$ 作用在 $\mathbf{v}$ 上只改变长度、不改变方向。

为什么重要：反复做同一个线性变换 $A^k\mathbf{x}$ 时，结果由**最大的特征值**决定。$|\lambda|_{\max} < 1$ 时会收敛到 0，$>1$ 时会爆炸。这就是：

- RNN 梯度消失/爆炸的原因；
- 值迭代收敛的原因（第 2 章：$\gamma P$ 的特征值绝对值都 $\le\gamma<1$）。

In [ ]:
A = torch.tensor([[0.9, 0.2], [0.1, 0.7]])
evals, evecs = torch.linalg.eig(A)
print("特征值:", evals.real)
x = torch.tensor([1.0, 1.0])
norms = []
for k in range(30):
    x = A @ x
    norms.append(x.norm().item())
plt.figure(figsize=(4, 2.5)); plt.semilogy(norms); plt.title("||A^k x|| (max |eig| < 1 -> decays)"); plt.xlabel("k"); plt.show()

---

## 5. 奇异值分解（SVD）与低秩近似

任何矩阵都可以分解为

$$
A = U\Sigma V^\top = \sum_{i=1}^r \sigma_i\, \mathbf{u}_i \mathbf{v}_i^\top,\qquad \sigma_1\ge\sigma_2\ge\dots\ge 0
$$

$\sigma_i$ 叫**奇异值**，表示第 $i$ 个"成分"的重要程度。只保留前 $k$ 个最大的奇异值，就得到**最优的秩 $k$ 近似**（Eckart–Young 定理）。

**LoRA 的思想**：微调时权重的变化量 $\Delta W$ 通常是"低秩"的，所以不去更新整个 $d\times d$ 矩阵，而是学两个小矩阵 $\Delta W = BA$，$B\in\mathbb{R}^{d\times r}$、$A\in\mathbb{R}^{r\times d}$，$r\ll d$。openpi 的 `pi0_fast_droid_lora` 等配置就是这样省显存的。

In [ ]:
torch.manual_seed(0)
# 构造一个"本质上是低秩 + 一点噪声"的矩阵
U0 = torch.randn(100, 3); V0 = torch.randn(3, 80)
M = U0 @ V0 + 0.1 * torch.randn(100, 80)
U, S, Vh = torch.linalg.svd(M, full_matrices=False)
plt.figure(figsize=(4, 2.5)); plt.plot(S, "o-", ms=3); plt.title("singular values"); plt.yscale("log"); plt.show()
print("前 5 个奇异值:", S[:5].round(decimals=2))   # 只有 3 个很大 -> 本质上是秩 3

---

## 练习

### 练习 1：余弦相似度矩阵

给定 `X (N, d)` 和 `Y (M, d)`，返回 `(N, M)` 的余弦相似度矩阵。这就是 CLIP 计算"每张图和每句话有多匹配"的方式。

In [ ]:
def cosine_sim_matrix(X, Y, eps=1e-8):
    # TODO: 先把每一行归一化成单位向量，然后做一次矩阵乘法
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def cosine_sim_matrix(X, Y, eps=1e-8):
    Xn = X / (X.norm(dim=-1, keepdim=True) + eps)   # (N, d)，每行长度为 1
    Yn = Y / (Y.norm(dim=-1, keepdim=True) + eps)   # (M, d)
    return Xn @ Yn.T                                # (N, M)

In [ ]:
# ✅ 自测
X = torch.randn(5, 16); Y = torch.randn(7, 16)
S = cosine_sim_matrix(X, Y)
assert S.shape == (5, 7)
assert torch.isclose(S[1, 2], torch.nn.functional.cosine_similarity(X[1], Y[2], dim=0), atol=1e-6)
assert torch.allclose(cosine_sim_matrix(X, X).diagonal(), torch.ones(5), atol=1e-6)
print("通过 ✅")

### 练习 2：最优低秩近似

用 SVD 实现 `low_rank_approx(M, k)`，返回 $M$ 的最优秩 $k$ 近似。然后观察误差如何随 $k$ 变化。

In [ ]:
def low_rank_approx(M, k):
    # TODO: U, S, Vh = torch.linalg.svd(M, full_matrices=False)，只保留前 k 个
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def low_rank_approx(M, k):
    U, S, Vh = torch.linalg.svd(M, full_matrices=False)
    # U[:, :k] (m,k) * S[:k] 广播到每一列 -> (m,k)，再乘 Vh[:k] (k,n) -> (m,n)
    return (U[:, :k] * S[:k]) @ Vh[:k]

In [ ]:
# ✅ 自测
for k in [1, 2, 3, 5, 10]:
    err = (M - low_rank_approx(M, k)).norm() / M.norm()
    print(f"k={k:2d} 相对误差 {err:.4f}")
assert torch.linalg.matrix_rank(low_rank_approx(M, 3)) == 3
assert (M - low_rank_approx(M, 3)).norm() / M.norm() < 0.1
print("通过 ✅ —— 秩 3 就几乎完美地还原了 M")

### 练习 3：LoRA 能省多少参数？

一个线性层权重 $W\in\mathbb{R}^{d_{out}\times d_{in}}$。全量微调要更新 $d_{out}d_{in}$ 个参数；LoRA 秩为 $r$ 时只更新 $r(d_{out}+d_{in})$ 个。写一个函数返回 LoRA 参数量占全量的比例。

In [ ]:
def lora_ratio(d_out, d_in, r):
    # TODO
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def lora_ratio(d_out, d_in, r):
    return r * (d_out + d_in) / (d_out * d_in)

In [ ]:
# ✅ 自测：Gemma-2B 的注意力投影大约是 2048x2048
ratio = lora_ratio(2048, 2048, 16)
print(f"r=16 时 LoRA 只需要全量参数的 {ratio*100:.2f}%")
assert abs(ratio - 0.015625) < 1e-9
print("通过 ✅")

---

## 小结

| 概念 | 一句话 | 在哪里用 |
|---|---|---|
| 点积 / 余弦 | 衡量方向一致程度 | 注意力、CLIP/SigLIP |
| 矩阵乘法 | 线性变换；形状"中间消掉" | 所有网络层 |
| 线性方程组 | `solve` 而不是求逆 | 贝尔曼方程闭式解 |
| 特征值 | 反复变换时决定收敛还是爆炸 | 值迭代收敛性、梯度消失 |
| SVD / 低秩 | 少数成分就能近似整体 | LoRA、模型压缩 |

**思考题**：注意力分数为什么要除以 $\sqrt{d}$？提示：如果 $\mathbf{q},\mathbf{k}$ 的每个分量都是独立的标准正态分布，$\mathbf{q}^\top\mathbf{k}$ 的方差是多少？（答案在 [1.05 概率与统计](1.05_probability.md) 的练习里验证。）